In [ ]:
import os
from pathlib import Path
from urllib.parse import quote_plus

import certifi
import pandas as pd
from dotenv import load_dotenv
from pymongo import MongoClient

load_dotenv(dotenv_path=".env", override=True)

In [ ]:
# Change these values for the CSV you want to upload.
csv_path = Path("data/product_category_name_translation_atlas.csv")
collection_name = "product_categories"
batch_size = 1_000

if not csv_path.is_file():
    raise FileNotFoundError(f"CSV file not found: {csv_path.resolve()}")
if not collection_name.strip():
    raise ValueError("collection_name cannot be empty")

In [ ]:
# Expected .env keys: hostnameMongoDBAtlas, databaseMongoDBAtlas, usernameMongoDBAtlas, passwordMongoDBAtlas
load_dotenv(dotenv_path=".env", override=True)

mongo_hostname = os.getenv("hostnameMongoDBAtlas")
mongo_database = os.getenv("databaseMongoDBAtlas")
mongo_username = os.getenv("usernameMongoDBAtlas")
mongo_password = os.getenv("passwordMongoDBAtlas")

mongo_config = {
    "hostnameMongoDBAtlas": mongo_hostname,
    "databaseMongoDBAtlas": mongo_database,
    "usernameMongoDBAtlas": mongo_username,
    "passwordMongoDBAtlas": mongo_password,
}
missing = [name for name, value in mongo_config.items() if not value]
if missing:
    raise ValueError(f"Missing .env values: {', '.join(missing)}")

mongo_hostname = mongo_hostname.removeprefix("mongodb+srv://").rstrip("/")
mongo_uri = (
    f"mongodb+srv://{quote_plus(mongo_username)}:{quote_plus(mongo_password)}"
    f"@{mongo_hostname}/{quote_plus(mongo_database)}"
    "?retryWrites=true&w=majority&appName=CSVUploader"
)

client = MongoClient(
    mongo_uri,
    tlsCAFile=certifi.where(),
    serverSelectionTimeoutMS=30_000,
)
client.admin.command("ping")
database = client[mongo_database]
collection = database[collection_name]
print(f"Connected to Atlas database: {mongo_database}")

In [ ]:
dataframe = pd.read_csv(csv_path, encoding="utf-8-sig")
# MongoDB stores null values, whereas pandas represents missing values as NaN.
dataframe = dataframe.astype(object).where(pd.notna(dataframe), None)

print(f"Loaded {len(dataframe):,} rows and {len(dataframe.columns)} columns")
display(dataframe.head())

In [ ]:
records = dataframe.to_dict(orient="records")
inserted_count = 0

try:
    for start in range(0, len(records), batch_size):
        batch = records[start : start + batch_size]
        if batch:
            result = collection.insert_many(batch, ordered=False)
            inserted_count += len(result.inserted_ids)
            print(f"Inserted {inserted_count:,}/{len(records):,} records")
finally:
    client.close()

print(f"Upload complete: {inserted_count:,} records inserted into {collection_name!r}")